In [1]:
import os
from openai import OpenAI, AsyncOpenAI
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
import asyncio
from pydantic import BaseModel, Field
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
load_dotenv(override=True)
from agents import GuardrailFunctionOutput, output_guardrail
import os
from openai import OpenAI, AsyncOpenAI
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
import asyncio
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
load_dotenv(override=True)


True

In [2]:
import openai
print(openai.__version__)

from agents import Agent
print("success")

3.0.0
success


In [3]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("Gemini_API")
# gemini = OpenAI(base_url = GEMINI_BASE_URL, api_key = google_api_key)

# response = gemini.chat.completions.create(model="gemini-3.5-flash-lite", messages=[{"role":"user", "content":"What is 2+2"}])
# print(response.choices[0].message.content)

In [4]:
gemini_client = AsyncOpenAI(base_url = GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.5-flash-lite", openai_client=gemini_client)

# agent = Agent(name="Jokester", instructions="You are a joke teller who keeps things within 10 words", model=gemini_model)

In [211]:
# result = await Runner.run(agent, "Tell a joke about autonomous AI agents")

In [212]:
# result.final_output

In [213]:
# result.to_input_list()

## Logging and observability

In [214]:

# with trace("Telling a joke"):
#     result = await Runner.run(agent, "Tell a joke about Autonomous AI agents")
# print(result.final_output)

## Streaming the outputs

In [215]:
# result = Runner.run_streamed(agent, input="Please tell me 5 jokes abount Agentic AI")
# async for event in result.stream_events():
#     if event.type=="raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
#         print(event.data.delta, end="", flush=True)

In [216]:
pushover_url = "https://api.pushover.net/1/messages.json"
# implementing pushover notifications
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
print(pushover_token[:4])

# @function_tool
# def push_tool(message:str)->str:
#     """Send the notification to user as a push notification"""
#     payload = {"user": pushover_user, "token": pushover_token, "message": message}
#     result = requests.post(pushover_url, data=payload).status_code
#     return f"Push sent with API status code {result}"

ad42


In [217]:
# push_tool

In [218]:
# push_tool.description

In [219]:
# notifier = Agent(name="Notifier", model=gemini_model, instructions="You notify the user upon request", tools=[push_tool])

In [220]:
# result = await Runner.run(notifier, "Notify the user pizza is here")
# print(result.final_output)

## Memory and using memory with the agents

In [221]:
# agent = Agent(name="assistant", model=gemini_model)

In [222]:
# session = SQLiteSession("12345")
# response = await Runner.run(agent, "Hi there my name is Rohit", session=session)
# print(response.final_output)
# response = await Runner.run(agent, "What is my name?", session=session)
# print(response.final_output)

In [223]:
EMAIL_SMTP_SERVER=os.getenv("EMAIL_SMTP_SERVER")
EMAIL_PASSWORD=os.getenv("EMAIL_PASSWORD")
EMAIL_ADDRESS=os.getenv("EMAIL_ADDRESS")
USE_EMAIL = False

In [224]:
import requests
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject:{subject}, text_body:{text_body}")

In [225]:
import smtplib
from email.message import EmailMessage
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"]=EMAIL_ADDRESS
    msg["Subject"]=subject
    msg.set_content(text_body)
    msg.add_alternative(html_body)

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.send_message(msg)

In [226]:
# send_message("Testing testing", "123", "<html><body><strong>Fingers</strong></body></html>")

# Agent orchestration using two steps:
## From code by multiple calls to Runner.run or by LLMs which allow LLM to orchestrate this

In [227]:
intro = """You are a sales working for CompliAI a company that provides Saas tool for ensuring SOC2 compliance
and preparing for audits powered by AI. You write short emails"""

instructions1 = intro + "your email style is professional serious and gravitas oriented"
instructions2 = intro+"Your email style is witty and humorous"
instructions3= intro+"Your email style is concise and to the point"

In [228]:
salesagent1 = Agent(name="Professional Agent Sales", model=gemini_model, instructions=instructions1)
salesagent2 = Agent(name="Witty sales agent", model=gemini_model, instructions=instructions2)
salesagent3 = Agent(name="Concise sales agent", model=gemini_model, instructions=instructions3)

In [229]:
# result = Runner.run_streamed(salesagent1, input ="write a cold sales email")
# async for event in result.stream_events():
#     if event.type=="raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
#          print(event.data.delta, end="", flush=True)

In [230]:
# # A manager of emails is to be made here

# decision = """You pick best sales cold email from options Imagine you are a customer and which email is
# the one you are most likely to respond to. DO not give an explanation respond with the best email only"""

# sales_picker = Agent(name="sales_picker", model=gemini_model, instructions=decision)

# ## Parallel way to work
# message = "Write a cold sales email"
# with trace("Sales selection workflow"):
#     results = await asyncio.gather(
#         Runner.run(salesagent2, message),
#         Runner.run(salesagent3, message)
#     )
#     outputs = [result.final_output for result in results]
#     emails= "Cold sales email: \n \n"+"\n Email \n \n".join(outputs)
#     best = await Runner.run(sales_picker, emails)
#     print(f"Best sales email: \n {best.final_output}")

## Orchestrating by LLMs

In [231]:
# no.1 is via tools
description = "Use this tool to write sales email. In the input just instruct it to write a sales email"

tool1=salesagent1.as_tool(tool_name="sales_email_writer1", tool_description=description)
tool2=salesagent2.as_tool(tool_name="sales_email_writer2", tool_description=description)
# SO now the LLM has become a tool with description
from agents import function_tool

@function_tool
def send_message_tool(subject: str, text_body: str, html_body: str) -> str:
    """Send an email or push notification to the user."""
    
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
        return f"Email sent with subject '{subject}'"

    push(f"Subject: {subject}\n{text_body}")
    return "Push notification sent"

tools=[tool1, tool2, send_message_tool]
tools

[FunctionTool(name='sales_email_writer1', description='Use this tool to write sales email. In the input just instruct it to write a sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000022C70C42D50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='sales_email_writer2', description='Use this tool to write sales email. In the input just instruct it to write a sales email', params_json_schema={'description': 'Default input schema fo

In [232]:
instructions = """You are a sales manager at complAI. Your goal is to find best cold email using the tools"""
task=""" Follow these steps:
1. Generate Drafts: use all three email generation tools to create drafts. DO not procede until all 3 emails are produced.
2. Evaluate and select the best email from the customer's POV
3. Use your send message tool to send the message to the user send only 1 email."""

sales_manager = Agent(name="sales manager", instructions=instructions, tools=tools, model=gemini_model)

In [233]:
# from agents.extensions.visualization import draw_graph

# draw_graph(sales_manager)

In [234]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

OPENAI_API_KEY is not set, skipping trace export


Push: Subject: Best Cold Email Selected for complAI
Subject: SOC 2 compliance friction / CompliAI

[Prospect Name],

Achieving and maintaining SOC 2 compliance often consumes hundreds of engineering hours and delays critical enterprise deals. 

CompliAI automates this burden. Our AI-powered platform integrates with your infrastructure to continuously gather evidence, draft policies, and remediate gaps—reducing audit preparation time by up to 80%. 

We have helped companies like yours achieve audit readiness in weeks, not months, without pulling engineering teams from core product development.

Do you have 10 minutes next Tuesday at 10:00 AM to discuss your current compliance roadmap?

Best regards,

[Your Name]  
Sales, CompliAI


OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [207]:
# platform.openai.com/traces

OPENAI_API_KEY is not set, skipping trace export


## Orchestrating by LLM handoffs so control passes along LLMs

In [140]:
from agents import ModelSettings
decision = """You pick best sales cold email from options Imagine you are a customer and which email is
the one you are most likely to respond to. DO not give an explanation respond with the best email only"""

## force an agent to use a tool this way:-
require_tools = ModelSettings(tool_choice = "required")
sales_sender = Agent(name="sales_picker", model=gemini_model, instructions=decision, tools=[send_message_tool], model_settings=require_tools)

OPENAI_API_KEY is not set, skipping trace export


In [46]:
instructions = """You are a sales manager at ComplAI.
 You get your sales team to draft emails and send them to a sales picker"""

task = """Follow these steps:
1. Generate drafts to generate the drafts for emails do not stop until the two drafts are generated.
2. Handoff to sales sender to choose the best email."""

tools = [tool1, tool2]
handoffs=[sales_sender]

In [50]:
result =  await Runner.run(sales_manager, task)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Push: Subject: CompliAI Cold Email Selection & Analysis
As the Sales Manager at complAI, I have evaluated both generated drafts and selected the winning email.

Draft 1 is professional and direct, focusing on ROI and engineering resources.
Draft 2 uses humor and strong pain-point resonance, making it much more engaging and likely to get a reply.

Selected Winning Email:
Subject: Less compliance crying, more code-shipping 🚀

Hey [First Name],

Let’s be honest: preparing for a SOC 2 audit is right up there with root canals and stepping on Legos in the barefoot pain index. Endless spreadsheets, policy-writing induced wrist cramps, and asking developers for screenshots they treat like state secrets. 

CompliAI automates the whole miserable process. 

Our AI handles the heavy lifting—writing policies, gathering evidence, and mapping controls—so your team can get back to building product instead of drowning in compliance busywork. 

Cut your audit prep time by 80%. Keep your hair intact. 

G

OPENAI_API_KEY is not set, skipping trace export
